In [2]:
# from google.colab import files
# uploaded = files.upload()

# import os, zipfile

# os.makedirs("/root/.kaggle", exist_ok=True)
# for fn in uploaded.keys():
#     os.rename(fn, "/root/.kaggle/kaggle.json")
# os.chmod("/root/.kaggle/kaggle.json", 0o600)

# #!/bin/bash
# !kaggle datasets download uciml/sms-spam-collection-dataset

# # Unzip the simulated roads accident dataset (it will produce those CSVs)
# with zipfile.ZipFile("/content/sms-spam-collection-dataset.zip", "r") as z:
#     z.extractall("sms-spam-collection-dataset")

In [4]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 53.1 MB/s eta 0:00:00


In [5]:
import pandas as pd
import numpy as np
import gensim
import pandas as pd
import re
import nltk

In [6]:
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [7]:
messages = pd.read_csv('/content/spamhamdata.csv', sep='\t', names=['label', 'message'])

In [8]:
messages[:2]

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...


In [12]:
corpus = []
for i in range(len(messages)):
    review = re.sub('[^a-zA-Z]', ' ', messages['message'][i])
    review = review.lower().split()
    review = [ps.stem(word) for word in review if word not in stopwords.words()]
    review = ' '.join(review)
    corpus.append(review)
corpus[:2]

['jurong point crazi avail bugi great world buffet amor', 'lar joke wif']

In [13]:
## Bag of words

In [14]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=2500, ngram_range=(1, 2))

In [16]:
X = cv.fit_transform(corpus).toarray()

In [20]:
y = pd.get_dummies(messages['label'])
y[:5]

,ham,spam
0,True,False
1,True,False
2,False,True
3,True,False
4,True,False


In [21]:
y = y.iloc[:,1].values
y

array([False, False,  True, ..., False, False, False])

In [22]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

In [23]:
from sklearn.naive_bayes import MultinomialNB
spam_detect_model = MultinomialNB().fit(X_train, y_train)

In [24]:
spam_detect_model.score(X_test, y_test)

0.9856502242152466

In [25]:
from sklearn.metrics import classification_report
print(classification_report(y_test, spam_detect_model.predict(X_test)))

              precision    recall  f1-score   support

       False       0.99      0.99      0.99       955
        True       0.96      0.94      0.95       160

    accuracy                           0.99      1115
   macro avg       0.98      0.97      0.97      1115
weighted avg       0.99      0.99      0.99      1115

